# Baseline Evaluation — Final (N=2000)
Loads saved scribbles from GDrive, runs full evaluation on Colab A100.

In [ ]:
## 1. Setup
import os, sys, json
import numpy as np
import torch
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from pathlib import Path
from IPython.display import display

if 'google.colab' in str(get_ipython()):
    import getpass
    !pip install -q diffusers transformers accelerate controlnet_aux scikit-learn peft
    from huggingface_hub import login
    from google.colab import userdata
    login(token=userdata.get('HF'))
    token = userdata.get('GITHUB') or getpass.getpass('GitHub token: ')
    repo = 'conditional-matching-paper'
    if not os.path.exists(repo):
        !git clone https://{token}@github.com/orineo1/conditional-matching-paper.git
    !cd {repo} && git checkout compareSDvsNaive
    for p in [f'/content/{repo}', f'/content/{repo}/SD_cond_SD_controlnet']:
        if p not in sys.path: sys.path.insert(0, p)

from models        import load_models
from clip_utils    import load_clip_model, encode_images_clip
from run_dps       import compute_clip_softmax
from metrics       import compute_mmd

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
## 2. Config — choose experiment

EXPERIMENT = 'SkewedTarget'   # ← change this: SkewedTarget | BalancedTarget | GenderInterpolation | AgeInterpolation

EXPERIMENT_CONFIGS = {
    'SkewedTarget': dict(
        best_jid         = 44430631,
        gdrive_runs_root = '/content/drive/MyDrive/conditional-matching/runs',
        seed             = 5,
        n_eval           = 2000,
        n_target         = 2000,
        neutral_prompt   = 'a superrealistic professional photograph of',
        controlnet_scale = 0.5,
        mode             = 'binary',
        groups = [
            dict(label='Man',   prompt='a superrealistic portrait photograph of a man, studio lighting',   frac=0.25, color='royalblue'),
            dict(label='Woman', prompt='a superrealistic portrait photograph of a woman, studio lighting', frac=0.75, color='crimson'),
        ],
    ),
    'BalancedTarget': dict(
        best_jid         = 44424933,
        gdrive_runs_root = '/content/drive/MyDrive/conditional-matching/runs',
        seed             = 5,
        n_eval           = 2000,
        n_target         = 2000,
        neutral_prompt   = 'a superrealistic professional photograph of',
        controlnet_scale = 0.5,
        mode             = 'binary',
        groups = [
            dict(label='Man',   prompt='a superrealistic portrait photograph of a man, studio lighting',   frac=0.5, color='royalblue'),
            dict(label='Woman', prompt='a superrealistic portrait photograph of a woman, studio lighting', frac=0.5, color='crimson'),
        ],
    ),
    'GenderInterpolation': dict(
        best_jid         = 44432053,
        gdrive_runs_root = '/content/drive/MyDrive/conditional-matching/runs',
        seed             = 5,
        n_eval           = 2000,
        n_target         = 2000,
        neutral_prompt   = 'a superrealistic professional photograph of',
        controlnet_scale = 0.5,
        mode             = 'multiclass',
        groups = [
            dict(label='Woman',                  prompt='superrealistic portrait photograph of a woman, extremely feminine features, studio lighting',                                                               frac=0.25, color='crimson'),
            dict(label='Woman w/ masc features', prompt='a superrealistic portrait photograph of a woman with masculine features, heavy brow ridge, studio lighting',                                               frac=0.25, color='orchid'),
            dict(label='Man w/ fem features',    prompt='a superrealistic portrait photograph of a man with extremely feminine feminine features, soft delicate face, high cheekbones, studio lighting',            frac=0.25, color='slategray'),
            dict(label='Man',                    prompt='a superrealistic portrait photograph of a man, extremely masculine features, studio lighting',                                                             frac=0.25, color='steelblue'),
        ],
    ),
    'AgeInterpolation': dict(
        best_jid         = 44492374,
        gdrive_runs_root = '/content/drive/MyDrive/conditional-matching/runs/InterpolationMenWomen',
        seed             = 42,
        n_eval           = 120,
        n_target         = None,
        neutral_prompt   = 'a superrealistic professional photograph of',
        controlnet_scale = 0.5,
        mode             = 'age',
        age_min          = 40,
        age_max          = 80,
        age_step         = 1,
        n_per_age        = 3,
    ),
}

cfg  = EXPERIMENT_CONFIGS[EXPERIMENT]
SEED = cfg['seed']
print(f'Experiment: {EXPERIMENT}  |  jid={cfg["best_jid"]}  |  mode={cfg["mode"]}')

In [ ]:
## 3. Mount Drive & Load Scribbles

from google.colab import drive
drive.mount('/content/drive')

jid      = cfg['best_jid']
run_dir  = Path(cfg['gdrive_runs_root']) / f'dps_main_{jid}'
base_dir = run_dir / 'baselines'

# Load all scribbles saved by the cluster script
scribble_names = ['source', 'avg', 'sdedit', 'sdedit_best', 'lgd_cm']
scribbles = {}
for name in scribble_names:
    p = base_dir / f'scribble_{name}.png'
    if p.exists():
        scribbles[name] = Image.open(p)
        print(f'  Loaded scribble_{name}.png')
    else:
        print(f'  WARNING: {p} not found — skipping')

# Load metadata
with open(base_dir / 'baselines_meta.json') as f:
    meta = json.load(f)
print(f"\nMeta: {meta['n_candidates']} candidates, best idx={meta['best_candidate_idx']}, best MMD={meta['best_candidate_mmd']:.5f}")

# Show all scribbles
n = len(scribbles)
fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
for ax, (name, img) in zip(axes, scribbles.items()):
    ax.imshow(img, cmap='gray'); ax.set_title(name); ax.axis('off')
plt.suptitle(f'All Scribbles — {EXPERIMENT}', fontweight='bold')
plt.tight_layout(); display(fig); plt.close()

In [ ]:
## 4. Load Models

architect, sprinter        = load_models(device)
clip_model, clip_processor = load_clip_model(device)
print('Models loaded.')

In [ ]:
## 5. Helpers

def gen_images(scribble_pil, prompt, n, seed=None):
    sprinter.vae.to(dtype=torch.float16)
    generator = torch.Generator(device=sprinter.device).manual_seed(seed) if seed is not None else None
    imgs = []
    with torch.no_grad():
        for start in range(0, n, 2):
            bs = min(2, n - start)
            imgs.extend(sprinter(
                prompt=[prompt]*bs, image=[scribble_pil]*bs,
                num_inference_steps=2, guidance_scale=0.0,
                controlnet_conditioning_scale=cfg['controlnet_scale'],
                output_type='pil', generator=generator,
            ).images)
    sprinter.vae.to(dtype=torch.float32)
    return imgs

def clip_embed(images):
    tensors = torch.cat([TF.to_tensor(img).unsqueeze(0) for img in images]).to(device)
    clip_model.to(device)
    with torch.no_grad():
        embs = encode_images_clip(tensors, clip_model, clip_processor)
    clip_model.to('cpu')
    return embs

def binomial_ci(n_success, n_total, z=1.96):
    p  = n_success / n_total
    se = np.sqrt(p * (1-p) / n_total)
    return p, p - z*se, p + z*se

print('Helpers ready.')

In [ ]:
## 6. Build Target Distribution (N=2000)

source_scribble = scribbles['source']
mode = cfg['mode']
n_target = cfg.get('n_target', 2000) or 2000

if mode in ('binary', 'multiclass'):
    all_target_imgs = []
    group_target_imgs = {}
    for i, g in enumerate(cfg['groups']):
        n_i = max(1, int(n_target * g['frac']))
        print(f"  [{g['label']}] n={n_i}...")
        imgs = gen_images(source_scribble, g['prompt'], n_i, seed=SEED + i * 1000)
        all_target_imgs.extend(imgs)
        group_target_imgs[g['label']] = imgs
    target_clip = clip_embed(all_target_imgs)

elif mode == 'age':
    ages      = list(range(cfg['age_min'], cfg['age_max'], cfg['age_step']))
    n_per_age = cfg['n_per_age']
    age_embs  = {}
    group_target_imgs = {}
    clip_model.to(device)
    from tqdm.notebook import tqdm
    with torch.no_grad():
        for age in tqdm(ages, desc='Age target'):
            prompt = (f'a superrealistic portrait photograph of a {age}-year-old man, '
                      'studio lighting, sharp focus, photographic')
            imgs = gen_images(source_scribble, prompt, n_per_age, seed=SEED + age * 7)
            embs = encode_images_clip(
                torch.cat([TF.to_tensor(img).unsqueeze(0) for img in imgs]).to(device),
                clip_model, clip_processor)
            age_embs[age] = embs.cpu()
            group_target_imgs[str(age)] = imgs
    clip_model.to('cpu')
    target_clip = torch.cat([age_embs[a] for a in ages], dim=0).to(device)

print(f'Target CLIP: {target_clip.shape}')

# Preview target groups
N_PREVIEW = 5
sample_groups = list(group_target_imgs.items())[:6]  # show max 6 groups
for label, imgs in sample_groups:
    n_cols = min(N_PREVIEW, len(imgs))
    fig, axes = plt.subplots(1, n_cols, figsize=(3*n_cols, 3))
    if n_cols == 1: axes = [axes]
    for ax, img in zip(axes, imgs[:n_cols]):
        ax.imshow(img); ax.axis('off')
    fig.suptitle(f'Target: {label}', fontweight='bold')
    plt.tight_layout(); display(fig); plt.close()

In [ ]:
## 7. Evaluate All Methods

N_EVAL   = cfg['n_eval']
N_PREVIEW = 5
results  = {}
all_imgs = {}
all_embs = {}

for name, scribble in scribbles.items():
    print(f'\n[{name}] generating {N_EVAL} images...')
    imgs = gen_images(scribble, cfg['neutral_prompt'], N_EVAL, seed=SEED)
    embs = clip_embed(imgs)
    mmd  = compute_mmd(embs, target_clip).item()

    if mode in ('binary', 'multiclass'):
        man_prompt   = cfg['groups'][0]['prompt'] if mode == 'binary' else cfg['groups'][-1]['prompt']
        woman_prompt = cfg['groups'][-1]['prompt'] if mode == 'binary' else cfg['groups'][0]['prompt']
        sr, _  = compute_clip_softmax(imgs, clip_model, clip_processor, man_prompt, woman_prompt, device)
        n_male = sum(1 for r in sr if r['label'] == 'male')
        p, lo, hi = binomial_ci(n_male, N_EVAL)
        results[name] = dict(mmd=mmd, p_male=p, ci_lo=lo, ci_hi=hi, n_male=n_male, n_female=N_EVAL-n_male)
    else:
        results[name] = dict(mmd=mmd)

    all_imgs[name] = imgs
    all_embs[name] = embs
    print(f'  MMD={mmd:.5f}' + (f"  p(male)={results[name].get('p_male', 'N/A'):.3f}" if mode != 'age' else ''))

print('\nDone.')

In [ ]:
## 8. Sample Images per Method

fig, axes = plt.subplots(len(scribbles), N_PREVIEW, figsize=(3*N_PREVIEW, 3*len(scribbles)))
for row, (name, imgs) in enumerate(all_imgs.items()):
    for col, img in enumerate(imgs[:N_PREVIEW]):
        axes[row, col].imshow(img); axes[row, col].axis('off')
    axes[row, 0].set_ylabel(name, fontsize=10)
plt.suptitle(f'Sample Images per Method — {EXPERIMENT}', fontweight='bold')
plt.tight_layout(); display(fig); plt.close()

In [ ]:
## 9. Results Table

source_mmd = results['source']['mmd']
rows = {}
for name, r in results.items():
    imp = (source_mmd - r['mmd']) / source_mmd * 100
    row = {'MMD': f"{r['mmd']:.5f}", 'Δ MMD vs source (%)': f'{imp:+.1f}%'}
    if mode != 'age':
        row.update({
            'p(male)':           f"{r['p_male']:.3f}",
            '95% CI':            f"[{r['ci_lo']:.3f}, {r['ci_hi']:.3f}]",
            'n_male / n_female': f"{r['n_male']} / {r['n_female']}",
        })
    rows[name] = row

df = pd.DataFrame(rows).T
display(df)

In [ ]:
## 10. Plots

names  = list(results.keys())
mmds   = [results[n]['mmd'] for n in names]
colors = ['#888888' if n == 'source' else '#2ca02c' if n == 'lgd_cm' else '#4C72B0' for n in names]

# MMD bar chart
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(names, mmds, color=colors, alpha=0.85)
ax.axhline(source_mmd, color='gray', linestyle='--', linewidth=1.0, label='source baseline')
for bar, mmd in zip(bars, mmds):
    imp = (source_mmd - mmd) / source_mmd * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0002,
            f'{imp:+.1f}%', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('MMD'); ax.set_title(f'MMD vs target — {EXPERIMENT} (N={N_EVAL})')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
plt.xticks(rotation=15, ha='right')
plt.tight_layout(); display(fig); plt.close()

# p(male) CI plot — binary/multiclass only
if mode != 'age':
    p_hats = [results[n]['p_male'] for n in names]
    ci_lo  = [results[n]['ci_lo']  for n in names]
    ci_hi  = [results[n]['ci_hi']  for n in names]
    target_frac = cfg['groups'][0]['frac']  # man frac

    fig, ax = plt.subplots(figsize=(8, 4))
    for i, (name, p, lo, hi, col) in enumerate(zip(names, p_hats, ci_lo, ci_hi, colors)):
        ax.errorbar(i, p, yerr=[[p-lo], [hi-p]], fmt='o', capsize=6,
                    linewidth=1.5, color=col, markersize=8)
    ax.axhline(target_frac, color='gray', linestyle='--', linewidth=0.8, label=f'target p={target_frac}')
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=15, ha='right')
    ax.set_ylabel('p(male)'); ax.set_title(f'Proportion male — 95% CI (N={N_EVAL})')
    ax.set_ylim(0, 1); ax.legend(); ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout(); display(fig); plt.close()

In [ ]:
## 11. Save Results to GDrive

out_dir = base_dir  # save alongside scribbles

# Table
df.to_csv(out_dir / f'results_{EXPERIMENT}.csv')
print(f'Saved results_{EXPERIMENT}.csv')

# MMD bar chart
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(names, mmds, color=colors, alpha=0.85)
ax.axhline(source_mmd, color='gray', linestyle='--', linewidth=1.0, label='source baseline')
for bar, mmd in zip(bars, mmds):
    imp = (source_mmd - mmd) / source_mmd * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0002,
            f'{imp:+.1f}%', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('MMD'); ax.set_title(f'MMD vs target — {EXPERIMENT} (N={N_EVAL})')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
fig.savefig(out_dir / f'mmd_bar_{EXPERIMENT}.png', dpi=150, bbox_inches='tight')
plt.close(); print(f'Saved mmd_bar_{EXPERIMENT}.png')

# Sample images per method
fig, axes = plt.subplots(len(scribbles), N_PREVIEW, figsize=(3*N_PREVIEW, 3*len(scribbles)))
for row, (name, imgs) in enumerate(all_imgs.items()):
    for col, img in enumerate(imgs[:N_PREVIEW]):
        axes[row, col].imshow(img); axes[row, col].axis('off')
    axes[row, 0].set_ylabel(name, fontsize=10)
plt.suptitle(f'Sample Images per Method — {EXPERIMENT}', fontweight='bold')
plt.tight_layout()
fig.savefig(out_dir / f'samples_{EXPERIMENT}.png', dpi=150, bbox_inches='tight')
plt.close(); print(f'Saved samples_{EXPERIMENT}.png')

print('\n✅ All results saved to GDrive.')